# NBA Data: Cleaning

This notebook takes the raw CSV files scraped from basketball-reference.com and turns them into clean tables that are ready for the database and the statistical analysis.

Here's the plan:

1. Load every file and take a first look: shape, missing values, duplicates and which seasons each file covers.
2. Drop columns that carry no information and give the same thing the same name in every table.
3. Fix formats: feet-inches to centimetres, pounds to kilograms, text dates to real dates, numbers stored as text to numbers.
4. Look for impossible values and decide what to do with outliers.
5. Fill missing values, each with a reason that comes from how basketball works.
6. Save everything into a `cleaned/` folder.

One thing to keep in mind: in sports data, an extreme value is usually a real performance, not a mistake. Stephen Curry's 2015-16 season is an outlier, and it's exactly what we want to study. So we don't cap outliers here, we flag the ones that come from tiny samples instead.

In [1]:
import os
import numpy as np
import pandas as pd

# Loading the data

Every scraped file is loaded into a dictionary, so we can loop over all of them with the same function.

`teams.csv` is left out on purpose: `team_seasons.csv` has exactly the same columns plus `club_id`, and it covers more seasons.

In [2]:
file_names = [
    "clubs",
    "team_seasons",
    "team_advanced",
    "champions",
    "coaches",
    "rosters",
    "per_game",
    "totals",
    "advanced",
    "play_by_play",
    "playoffs_per_game",
    "mvp_votes"
]

tables = {}

for name in file_names:
    tables[name] = pd.read_csv(name + ".csv")

# First look

For every table: its shape, how many values are missing in each column and how many rows are exact duplicates.

In [3]:
pd.set_option("display.max_columns", None)


def first_look(data, name):

    print("=====", name, "=====")
    print("Shape:", data.shape)
    print("Duplicate rows:", data.duplicated().sum())

    missing = data.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("Missing values:")
        print(missing)
    else:
        print("No missing values")

    print()

In [4]:
for name in tables:
    first_look(tables[name], name)

===== clubs =====
Shape: (30, 14)
Duplicate rows: 0
No missing values

===== team_seasons =====
Shape: (240, 8)
Duplicate rows: 0
No missing values

===== team_advanced =====
Shape: (240, 31)
Duplicate rows: 0
Missing values:
DUMMY               240
attendance            4
attendance_per_g      4
dtype: int64

===== champions =====
Shape: (8, 2)
Duplicate rows: 0
No missing values

===== coaches =====
Shape: (262, 28)
Duplicate rows: 0
Missing values:
dum-1      262
dum-2      262
dum-3      262
cur_g_p    134
cur_w_p    134
cur_l_p    134
fr_g_p      73
fr_w_p      73
fr_l_p      73
car_g_p     57
car_w_p     57
car_l_p     57
dtype: int64

===== rosters =====
Shape: (5141, 12)
Duplicate rows: 0
Missing values:
college    659
dtype: int64

===== per_game =====
Shape: (5758, 33)
Duplicate rows: 0
Missing values:
fg_pct       43
fg3_pct     355
fg2_pct      93
efg_pct      43
ft_pct      371
awards     5326
dtype: int64

===== totals =====
Shape: (5758, 34)
Duplicate rows: 0
Missing val

# Which seasons does each file cover?

The project needs the seasons from 2019-20 to 2023-24 (2020 to 2024 in the files), so every table should at least cover those. Remember that basketball-reference names a season by the year it ends: `2024` means 2023-24.

In [5]:
for name in tables:

    seasons = sorted(tables[name]["season"].unique()) if "season" in tables[name].columns else []

    if len(seasons) > 0:
        print(name, ":", seasons[0], "to", seasons[-1], "-", len(seasons), "seasons")
    else:
        print(name, ": no season column")

clubs : no season column
team_seasons : 2019 to 2026 - 8 seasons
team_advanced : 2019 to 2026 - 8 seasons
champions : 2019 to 2026 - 8 seasons
coaches : 2019 to 2026 - 8 seasons
rosters : 2019 to 2026 - 8 seasons
per_game : 2019 to 2026 - 8 seasons
totals : 2019 to 2026 - 8 seasons
advanced : 2019 to 2026 - 8 seasons
play_by_play : 2019 to 2026 - 8 seasons
playoffs_per_game : 2019 to 2026 - 8 seasons
mvp_votes : 2019 to 2026 - 8 seasons


Two problems show up here:

- **`per_game` and `advanced` only cover 2016.** The scraping loop stopped after the first season (the advanced table id on the site is not `advanced_stats`). `per_game` is not a problem, because every per-game number is just a total divided by games played, so we rebuild it from `totals` further down. **`advanced` has to be scraped again**; there's no way to compute PER, BPM or Win Shares from the other files.
- `team_seasons`, `champions` and `rosters` stop at 2025, while the other files go up to 2026. That's fine for the project questions, but worth knowing before joining tables.

# Dropping empty columns

The site uses empty spacer columns between groups of stats (`DUMMY`, `dum-1`, ...). They are 100% empty. The `ranker` column is just the row number on the web page, so it goes too.

In [6]:
def drop_useless_columns(data):

    empty_columns = [col for col in data.columns if data[col].isnull().all()]

    data = data.drop(columns=empty_columns)

    if "ranker" in data.columns:
        data = data.drop(columns=["ranker"])

    return data

In [7]:
for name in tables:
    tables[name] = drop_useless_columns(tables[name])

# Giving the same thing the same name

The scraped column names come straight from the website, so the same idea has different names in different tables: the player name is `name_display` in one place and `player` in another, the team is `team_name_abbr`, `team_id` or `team`, and the generic `id` column means a player, a coach or a team depending on the file. We fix that, so joins are easy later.

In [8]:
rename_rules = {
    "totals": {
        "name_display": "player",
        "team_name_abbr": "team_abbr",
        "games": "g",
        "games_started": "gs",
        "id": "player_id"
    },
    "per_game": {
        "name_display": "player",
        "team_name_abbr": "team_abbr",
        "games": "g",
        "games_started": "gs"
    },
    "advanced": {
        "name_display": "player",
        "team_name_abbr": "team_abbr",
        "games": "g",
        "games_started": "gs"
    },
    "play_by_play": {
        "name_display": "player",
        "team_name_abbr": "team_abbr",
        "games": "g",
        "games_started": "gs",
        "pct_1": "pct_pg",
        "pct_2": "pct_sg",
        "pct_3": "pct_sf",
        "pct_4": "pct_pf",
        "pct_5": "pct_c",
        "id": "player_id"
    },
    "playoffs_per_game": {
        "team_id": "team_abbr",
        "id": "player_id"
    },
    "mvp_votes": {
        "team_id": "team_abbr",
        "id": "player_id"
    },
    "coaches": {
        "team": "team_abbr",
        "id": "coach_id"
    },
    "team_advanced": {
        "team": "team_name",
        "id": "team_abbr"
    },
    "champions": {
        "champion": "champion_abbr"
    },
    "rosters": {
        "flag": "birth_country"
    }
}

In [9]:
for name in rename_rules:
    tables[name] = tables[name].rename(columns=rename_rules[name])

# Fixing formats

## Rosters: height, weight, birth date, country and experience

- Height is stored as text in feet-inches (`6-11`). We turn it into centimetres: one foot is 12 inches and one inch is 2.54 cm.
- Weight is in pounds. One pound is 0.4536 kg.
- `birth_date` is text like `May 22, 1991`. We turn it into a real date, so age can be computed.
- The country column looks like `us US`. We keep only the upper-case code.
- In `years_experience`, rookies are marked `R`. A rookie has zero seasons of experience before this one, so `R` becomes `0` and the whole column becomes a number.

In [10]:
def height_to_cm(height):

    feet, inches = height.split("-")

    return round((int(feet) * 12 + int(inches)) * 2.54, 1)

In [11]:
rosters = tables["rosters"]

rosters["height_cm"] = rosters["height"].apply(height_to_cm)

rosters["weight_kg"] = round(rosters["weight"] * 0.4536, 1)

rosters["birth_date"] = pd.to_datetime(rosters["birth_date"], format="%B %d, %Y")

rosters["birth_country"] = rosters["birth_country"].str.split(" ").str[-1]

rosters["years_experience"] = rosters["years_experience"].replace("R", "0").astype(int)

rosters = rosters.drop(columns=["height", "weight"])

tables["rosters"] = rosters

In [12]:
tables["rosters"][["player", "height_cm", "weight_kg", "birth_date", "birth_country", "years_experience"]].head()

,player,height_cm,weight_kg,birth_date,birth_country,years_experience
0,Giannis Antetokounmpo,210.8,113.4,1994-12-06,GR,5
1,Eric Bledsoe,185.4,97.1,1989-12-09,US,8
2,Malcolm Brogdon,193.0,103.9,1992-12-11,US,2
3,Sterling Brown,195.6,99.3,1995-02-10,US,1
4,Isaiah Canaan,182.9,90.7,1991-05-21,US,5


## Team advanced: playoff marker and attendance

- Team names end with `*` when the team made the playoffs (`San Antonio Spurs*`). We move that information into its own `made_playoffs` column (1 / 0) and clean the name.
- Attendance is stored as text with commas (`756,445`). We remove the commas and turn it into a number.

In [13]:
team_advanced = tables["team_advanced"]

team_advanced["made_playoffs"] = team_advanced["team_name"].str.endswith("*").astype(int)

team_advanced["team_name"] = team_advanced["team_name"].str.replace("*", "", regex=False).str.strip()

for col in ["attendance", "attendance_per_g"]:
    team_advanced[col] = team_advanced[col].str.replace(",", "", regex=False).astype(float)

tables["team_advanced"] = team_advanced

## Clubs: founding year

`year_min` is the first season of the club, written like `1949-50`. The club was founded in the first year of that season, so we keep `1949` as a number.

In [14]:
clubs = tables["clubs"]

clubs["founded_year"] = clubs["year_min"].str[:4].astype(int)

clubs = clubs.drop(columns=["year_min", "year_max"])

tables["clubs"] = clubs

## MVP votes: tied ranks

When two players get the same number of points, the site writes the rank as `7T` (tied for 7th). That makes the whole column text. We keep the tie information in an `is_tied` column and turn the rank into a number.

In [15]:
mvp_votes = tables["mvp_votes"]

mvp_votes["is_tied"] = mvp_votes["rank"].str.endswith("T").astype(int)

mvp_votes["rank"] = mvp_votes["rank"].str.replace("T", "", regex=False).astype(int)

tables["mvp_votes"] = mvp_votes

# Rebuilding per_game from totals

Every per-game number is a total divided by games played, so we can rebuild the full `per_game` table for all seasons from `totals`. The percentage columns don't depend on the number of games, so they are copied as they are.

In [16]:
totals = tables["totals"]

count_columns = [
    "mp", "fg", "fga", "fg3", "fg3a", "fg2", "fg2a", "ft", "fta",
    "orb", "drb", "trb", "ast", "stl", "blk", "tov", "pf", "pts"
]

pct_columns = ["fg_pct", "fg3_pct", "fg2_pct", "efg_pct", "ft_pct"]

per_game = totals[["season", "player", "player_id", "age", "team_abbr", "pos", "g", "gs"] + pct_columns].copy()

for col in count_columns:
    per_game[col + "_per_g"] = round(totals[col] / totals["g"], 1)

tables["per_game"] = per_game

In [17]:
tables["per_game"].shape

(5758, 31)

# Impossible values

Some values simply can't be real: a shooting percentage above 100%, more games than a season has, a player shorter than 1.60 m. We count them for every table.

Two exceptions are not mistakes: `efg_pct` and `ts_pct` can go above 1. A player who takes one shot all season and makes a three gets an eFG% of 1.5. That's why their upper limit is higher.

In [18]:
def check_invalid(data, col, low, high=None):

    if high is None:
        invalid = data[col] < low
    else:
        invalid = (data[col] < low) | (data[col] > high)

    print(col, ":", invalid.sum())

In [19]:
invalid_rules = {
    "rosters": [
        ("height_cm", 160, 235),
        ("weight_kg", 60, 160),
        ("years_experience", 0, 25)
    ],
    "totals": [
        ("age", 18, 45),
        ("g", 1, 82),
        ("fg_pct", 0, 1),
        ("fg3_pct", 0, 1),
        ("ft_pct", 0, 1),
        ("efg_pct", 0, 1.5)
    ],
    "play_by_play": [
        ("pct_pg", 0, 100),
        ("pct_sg", 0, 100),
        ("pct_sf", 0, 100),
        ("pct_pf", 0, 100),
        ("pct_c", 0, 100)
    ],
    "team_advanced": [
        ("wins", 0, 82),
        ("losses", 0, 82),
        ("ts_pct", 0, 1)
    ],
    "mvp_votes": [
        ("award_share", 0, 1)
    ]
}

In [20]:
for name in invalid_rules:

    print("=====", name, "=====")

    for col, low, high in invalid_rules[name]:
        check_invalid(tables[name], col, low, high)

===== rosters =====
height_cm : 0
weight_kg : 0
years_experience : 0
===== totals =====
age : 0
g : 2
fg_pct : 0
fg3_pct : 0
ft_pct : 0
efg_pct : 0
===== play_by_play =====
pct_pg : 0
pct_sg : 0
pct_sf : 0
pct_pf : 0
pct_c : 0
===== team_advanced =====
wins : 0
losses : 0
ts_pct : 0
===== mvp_votes =====
award_share : 0


Only two values break a rule, both in `g` (games played): more than 82 games in one season.

In [21]:
tables["totals"][tables["totals"]["g"] > 82][["season", "player", "team_abbr", "g"]]

,season,player,team_abbr,g
2895,2023,Mikal Bridges,2TM,83
3658,2024,Buddy Hield,2TM,84


These are not mistakes. Both are `2TM` rows, meaning the player was traded mid-season. His new team had played fewer games than his old one at the time of the trade, so he ended up playing more than 82 games in total. That's a real basketball situation, so we keep these values as they are.

Everything else is within range. The scraped data matches the website exactly, so this is what we'd expect, but it's worth checking before trusting it.

# Outliers: flag, don't cap

Look at the highest PER values of 2015-16. Stephen Curry had one of the best seasons ever, and he's not even on top: a few players with 3 to 6 minutes for the whole season are above him.

In [22]:
tables["advanced"].sort_values("per", ascending=False)[["player", "mp", "per", "ts_pct", "bpm"]].head(6)

,player,mp,per,ts_pct,bpm
706,Zhou Qi,1,80.4,1.000,52.0
2872,Ahmad Caver,1,76.2,1.000,36.9
3553,Stanley Umude,2,65.6,0.532,48.6
496,Greg Monroe,5,60.2,0.600,25.2
4282,Markquis Nowell,4,55.5,0.532,43.3
2062,Udonis Haslem,3,54.6,1.000,31.1


Those values are real (the player really did score on his only shots), but they come from a tiny sample, so they say nothing about how good the player is. Capping them with the IQR rule would be wrong twice: it would also cap the real stars, who are exactly what the project is about.

So instead of changing any value, we add a `Low_Minutes_FG` column: 1 if the player played fewer than 500 minutes in that season. Before any ranking, filter on this flag. 500 minutes is roughly 6 games' worth of a full season; feel free to change the threshold, but write down why.

For playoffs the bar is lower, because a playoff run is only 4 to 28 games: we use 50 minutes.

In [23]:
def add_low_minutes_flag(data, minutes_col, threshold):

    data["Low_Minutes_FG"] = 0

    data.loc[data[minutes_col] < threshold, "Low_Minutes_FG"] = 1

    print("Flagged rows:", data["Low_Minutes_FG"].sum(), "of", len(data))

    return data

In [24]:
tables["totals"] = add_low_minutes_flag(tables["totals"], "mp", 500)

tables["advanced"] = add_low_minutes_flag(tables["advanced"], "mp", 500)

tables["play_by_play"] = add_low_minutes_flag(tables["play_by_play"], "mp", 500)

tables["per_game"]["mp"] = tables["totals"]["mp"]
tables["per_game"] = add_low_minutes_flag(tables["per_game"], "mp", 500)

Flagged rows: 2359 of 5758
Flagged rows: 2359 of 5758
Flagged rows: 2359 of 5758
Flagged rows: 2359 of 5758


In [25]:
playoffs = tables["playoffs_per_game"]

playoffs["mp"] = round(playoffs["g"] * playoffs["mp_per_g"])

tables["playoffs_per_game"] = add_low_minutes_flag(playoffs, "mp", 50)

Flagged rows: 548 of 1765


# Traded players

A player traded during the season has one row per team plus one total row, where the team is `2TM`, `3TM` or `4TM`. We mark the total rows, so it's easy to choose the right ones later:

- For season-wide rankings, keep the total rows (and the rows of players who stayed with one team).
- For questions about one team, keep the team rows.

In [26]:
def add_total_row_flag(data):

    data["Is_Total_Row"] = data["team_abbr"].isin(["2TM", "3TM", "4TM"]).astype(int)

    return data

In [27]:
for name in ["totals", "per_game", "advanced", "play_by_play"]:
    tables[name] = add_total_row_flag(tables[name])

print("Total rows in totals:", tables["totals"]["Is_Total_Row"].sum())

Total rows in totals: 623


# Filling missing values

Every missing value here has a basketball reason, so we fill them one by one instead of using the median everywhere.

**Shooting percentages.** A player who never attempted a three has no 3P% (0 makes / 0 attempts). We fill these with 0, but before that we keep a `*_Missing_Flag` column. That matters: when you average 3P%, filter out the flagged rows, otherwise the zeros pull the average down.

In [28]:
def add_missing_flags(data, columns):

    for col in columns:

        if col in data.columns and data[col].isnull().any():
            data[col + "_Missing_Flag"] = data[col].isnull().astype(int)

    return data


def fill_zero(data, columns):

    for col in columns:

        if col in data.columns:
            data[col] = data[col].fillna(0)

    return data

In [29]:
shooting_columns = ["fg_pct", "fg3_pct", "fg2_pct", "efg_pct", "ft_pct", "ts_pct", "fg3a_per_fga_pct", "fta_per_fga_pct", "tov_pct"]

for name in ["totals", "per_game", "advanced", "playoffs_per_game", "mvp_votes"]:
    tables[name] = add_missing_flags(tables[name], shooting_columns)
    tables[name] = fill_zero(tables[name], shooting_columns)

**Coaches' playoff columns.** Empty when the coach didn't coach a playoff game, in that season (`cur_*_p`), with that franchise (`fr_*_p`) or in his whole career (`car_*_p`). Zero playoff games is the true value, so we fill with 0.

In [30]:
playoff_columns = [col for col in tables["coaches"].columns if col.endswith("_p")]

tables["coaches"] = fill_zero(tables["coaches"], playoff_columns)

**Attendance.** The four empty values are all in the 2020-21 season. Because of COVID, many arenas had no fans that season, so 0 is the real attendance.

In [31]:
tables["team_advanced"][tables["team_advanced"]["attendance"].isnull()][["season", "team_name"]]

,season,team_name
75,2021,Miami Heat
76,2021,Indiana Pacers
83,2021,Sacramento Kings
89,2021,Oklahoma City Thunder


In [32]:
tables["team_advanced"] = fill_zero(tables["team_advanced"], ["attendance", "attendance_per_g"])

**Plus-minus.** Empty for two players who played a single minute all season. With no time on the court, the team's point difference while they played is 0.

In [33]:
tables["play_by_play"] = fill_zero(tables["play_by_play"], ["plus_minus_on", "plus_minus_net"])

**Text columns.** An empty `awards` cell means the player won no award that season, and an empty `college` means the player came to the NBA without playing college basketball (most international players). We write that explicitly instead of leaving it empty.

In [34]:
def fill_text(data, col, value):

    if col in data.columns:
        data[col] = data[col].fillna(value)

    return data

In [35]:
for name in ["totals", "per_game", "advanced", "play_by_play"]:
    tables[name] = fill_text(tables[name], "awards", "No Award")

tables["rosters"] = fill_text(tables["rosters"], "college", "No College")

# Players table

As planned for the database, the fixed information about each player (things that don't change from season to season) gets its own table, built from the rosters. Height and weight stay in the rosters, because they can change between seasons.

In [36]:
players = tables["rosters"][["player_id", "player", "birth_date", "birth_country"]]

players = players.drop_duplicates(subset="player_id")

tables["players"] = players

print("Players:", len(players), "| unique ids in rosters:", tables["rosters"]["player_id"].nunique())

Players: 1279 | unique ids in rosters: 1279


# Final check

Missing values should be gone everywhere, except in tables we chose to leave alone.

In [37]:
for name in tables:

    missing = tables[name].isnull().sum().sum()

    print(name, ":", tables[name].shape, "| missing values:", missing)

clubs : (30, 13) | missing values: 0
team_seasons : (240, 8) | missing values: 0
team_advanced : (240, 30) | missing values: 0
champions : (8, 2) | missing values: 0
coaches : (262, 25) | missing values: 0
rosters : (5141, 12) | missing values: 0
per_game : (5758, 39) | missing values: 0
totals : (5758, 40) | missing values: 0
advanced : (5758, 36) | missing values: 0
play_by_play : (5758, 28) | missing values: 0
playoffs_per_game : (1765, 38) | missing values: 0
mvp_votes : (93, 24) | missing values: 0
players : (1279, 4) | missing values: 0


# Saving

Every cleaned table goes into the `cleaned/` folder, with the same name as the raw file. `players.csv` is new.

In [38]:
os.makedirs("cleaned", exist_ok=True)

for name in tables:
    tables[name].to_csv(os.path.join("cleaned", name + ".csv"), index=False)

print("Saved", len(tables), "files to cleaned/")

Saved 13 files to cleaned/
